# pyamapping - mapping functions for audio computing (and beyond)

by Thomas Hermann and Dennis Reinsch, 2023++

## Introduction 

### Background / History

- pyamapping bundles frequently used mapping functions for audio computing
- the earliest functions were reimplementations of ampdb, dbamp, midicps, cpsmidi, linlin, coded in analogy to SuperCollider3 functions to be used within the sc3nb package. (coded by TH)
- later, when I started pya, the same functions were needed, yet importing sc3nb would have caused unwanted dependencies, so we created pyamapping as a very lean package that both sc3nb and pya depend on (created by DR)
- now in 2025 pyamapping grows strongly (additions by TH) 
  - firstly by adding many mapping functions available in sc3 which were beforehand not copied
  - secondly, by introducing ChainableArray, a class that wraps numpy.ndarrays, allowing to daisy chain operations on numpy arrays, similar to how we offer it for pya.
- This notebook introduces the available mapping functions with examples.

### Overview

**pyamapping** offers a set of mapping functions often used 

- in the context of sound and computer music 
- in the context of auditory display and sonification (e.g. parameter mapping sonifications)
- ...

A source of inspiration is librosa and Supercollider3. This package reimplements them and adds mappings used in the interactive sonification stack (cf. <https://github.com/interactive-sonification>), including the following packages that all make use of pyamapping:

- **sc3nb** - sc3 interface for Python and Jupyter notebooks
- **pya**  - the python Audio Coding Package
- **mesonic** - a middleware for sonification and auditory display 
- **sonecules** - a high-level class library for sonification and auditory display

**Chainable numpy arrays**

Method chaining offers concise syntax and proved helpful in pya.
Numpy offers method chaining only for few functions.
This package extends method chaining 

- by inheriting the class `ChainableArray` from `numpy.ndarray`
- by adding wrappers to enable a method chain syntax for most `ufuncs`
- by providing a general `map()` method for direct vectorized mapping
- by providing helper functions to vectorize Python functions into methods

Furthermore it offers some convenience functions, e.g.

- to plot arrays (optionally as signal with given sample rate)

We hope that pyamapping will help to write signal transformations and manipulations in a more concise, compact and readible manner.

**Imports and Headers**

- as pyamapping is a long name, importing as `pam` is a suggested abbreviation 
- matplotlib and pprint imports are merely for showing example output

In [ ]:
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
from pprint import pprint
import pyamapping as pam
from pyamapping import chain

mpl.rcParams['figure.figsize'] = (9, 2.5)

## Available pyamapping functions - Overview

- in import of pyamapping, wrappers are automatically created for numpy ufuncs.
  - later versions may have them created verbatim to enable code completion
- the following code simply lists those numpy functions plus special dedicated/ new pyamapping functions that do not have their origin in numpy

In [ ]:
from pyamapping.mappings import _list_numpy_ufuncs, pyamapping_functions

# print (i) a compact list of all unary and binary numpy functions 
# and (ii) all pyamapping functions
u_lists = [[], []]

for ufunc in _list_numpy_ufuncs():
    u_lists[ufunc.nin - 1].append(ufunc.__name__)

# compact list numpy functions
for i, li in enumerate(u_lists):
    print(f"\n=== numpy functions with {i+1} argument ===")
    pprint(li, compact=True, width=80)

# compact list of pyamapping functions
li = [el.__name__ for el in pyamapping_functions]
print(f"\n=== pyamapping functions: ===")
pprint(li, compact=True, width=80)

## pyamapping - Demonstration and Examples

### ChainableArray - Basics

Any numpy array can be turned into a chainable array by using the `ChainableArray` class defined in `pyamapping`.
- the chain() function provides a shortcut, making this construction shorter.


In [ ]:
from pyamapping import ChainableArray, chain

# some data
data = np.random.random(100)

# create ChainableArray
dch = ChainableArray(data)

# the same can be obtained shorter by
dch = chain(data)

ChainableArray offer the following methods:

- `to_array` - back to numpy ndarray
- `to_asig` - convert into pya.Asig
- `plot` - plot signal(s) as time series
- `mapvec` - map function on self by using numpy.vectorize
- `map` - apply function directly to the array itself

Here is a quick demonstration:

- let us
  - map $x \to (5x)^2 + 0.1$, 
  - plot as signal assuming sampling rate 100 Hz, 
  - convert into decibel, 
  - turn that into an audio signal (i.e. pya.Asig) 
  - and plot it in the same figure created above.
  - finally transform the ChainableArray back to a regular numpy.ndarray.

Using pyamapping, the code is both shorter and more concise than the above description:

In [ ]:
dch2 = dch.map(lambda x: (5*x)**2+0.1).plot(sr=100, color="r").mapvec(pam.amp_to_db)
a1 = dch2.to_asig(sr=100).plot(color="c", lw=0.8)

# conversion back to numpy array is rarely needed but if...
dd = dch2.to_array()
type(dd)

ChainableArray is a recent addition to pyamapping, yet introduced here as it makes demonstrations of mapping functions extremely readable...

## Tour of Mapping Functions: Functions from SuperCollider3 and librosa

- The following function have been implemented in analogy to their versions in Supercollider3 resp. librosa.
- Please note that some defaults may be different, e.g. functions extrapolate by default
- For the following demonstrations, we use `xs` to refer to the input array and `ys` for the outputs.

### `linlin`

- linlin is implemented in analogy to the SC3 linlin
- `linlin(v, x1, x2, y1, y2)` maps values v (scalar or arraylike) affine linearly so that [x1, x2] is mapped to [y1, y2]:
  $$ z = y_1 + \frac{v - x_1}{x_2 - x_1} \cdot (y_2 - y_1) $$
- note that this linlin function extrapolates by default
- clipping can be controlled via the clip argument (values None (default), "min", "max", or anything else for "minmax")
- A frequently used invocation is with $x_1 < x_2$, i.e. thinking of them as a range $[x_1, x_2]$

In [ ]:
pam.linlin(7, 0, 10, 100, 300)

In [ ]:
pam.linlin(7, 0, 5, 100, 300, "max") # clip result to maximum input range

- ChainableArray.linlin uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(0, 1, 100)) # turn data into ChainableArray
xs.plot("k-", label="input data") # plot input data
xs.linlin(0, 1, 1, 3).plot("r-", label="linlin")
xs.linlin(0.2, 0.7, -2, 2, "minmax").plot("g-", label="linlin with clip")
plt.legend(); plt.grid()

In [ ]:
# plot mapping function (output vs input)
ys = xs.linlin(0.25, 0.75, -1, 1, "minmax")
plt.figure(figsize=(3, 1.5)); plt.plot(xs, ys); plt.grid(); 

### `linexp`

- linexp is implemented in analogy to the SC3 linexp
- `linexp(v, x1, x2, y1, y2)` maps values v (scalar or arraylike) exponentially so that [x1, x2] is mapped to [y1, y2]:
  $$ z = y_1 \text{exp}\left(\frac{v - x_1}{x_2 - x_1}\cdot (\log(y_2) - \log(y_1))\right)
- note that this linexp function extrapolates by default
- clipping can be controlled via the clip argument (values None (default), "min", "max", or anything else for "minmax")
- A frequently used invocation is with $x_1 < x_2$, i.e. thinking of them as a range $[x_1, x_2]$

In [ ]:
pam.linexp(5, 1, 8, 2, 256)

In [ ]:
pam.linexp(7, 0, 5, 100, 300, "max") # clip result to maximum input range

- ChainableArray.linexp uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(0, 1, 100)) # turn data into ChainableArray
xs.plot("k-", label="input data") # plot input data
xs.linexp(0, 1, 0.01, 1).plot("r-", label="linexp")
xs.linexp(0.2, 0.7, 2, 0.2, "minmix").plot("g-", label="linexp with clip")
plt.legend(); plt.grid()

In [ ]:
# plot linexp mapping function (output vs input)
ys = xs.linexp(0.25, 0.85, 0.2, 2)
plt.figure(figsize=(3, 1.5)); plt.plot(xs, ys); plt.grid(); 
plt.plot(xs, ys); 

### `explin`

- explin is implemented in analogy to the SC3 function
- `explin(v, x1, x2, y1, y2)` maps values v (scalar or arraylike) logarithmically so that [x1, x2] is mapped to [y1, y2]:

    $$ y = y_1 + (y_2-y_1) \frac{\log(v / x_1)}{\log(x_2 / x_1)} $$

- note that this `explin` function extrapolates by default
- clipping can be controlled via the clip argument (values None (default), `"min"`, `"max"`, or anything else for `"minmax"`)
- A frequently used invocation is with $x_1 < x_2$, i.e. thinking of them as a range $[x_1, x_2]$

In [ ]:
# example: unmap a frequency to MIDI note with explin
f = 220 * 2**(-5/12) # 5 semitones higher than 220 Hz
pam.explin(f, 220, 440, 0, 12)


In [ ]:
# example: unmap amplitude to level in decibel
pam.explin(0.01, 0.001, 1.0, -30, 0, "max") # clip result to maximum input range

- ChainableArray.explin uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(0.01, 1, 100)) # turn data into ChainableArray
xs.plot("k-", label="input data") # plot input data
xs.explin(0.1, 1, 0, 1).plot("r-", label="explin")
xs.explin(0.1, 0.5, 1, 0, "minmix").plot("g-", label="explin with clip")
plt.legend(); plt.grid()

In [ ]:
# plot linexp mapping function (output vs input)
ys = xs.explin(0.01, 1, 0, 20)
plt.figure(figsize=(3, 1.5)); plt.plot(xs, ys); plt.grid(); 
plt.plot(xs, ys); 

### `lincurve`

- lincurve is implemented in analogy to the SC3 function
- `lincurve(v, x1, x2, y1, y2, curve=-2)` maps v (scalar or arraylike) from [x1, x2] to [y1, y2] using the following function (with c=curve): 

$$ y_1 + \frac{y_2 - y_1}{1.0 - e^c} \left(1 - \exp\left(c \frac{v - x_1}{x_2 - x_1}\right) \right) $$
- in contrast to `explin` (resp. `linexp`) this allows source (resp. target) range to include 0.
- note that this `lincurve` function extrapolates by default
- clipping can be controlled via the clip argument (values None (default), `"min"`, `"max"`, or anything else for `"minmax"`)
- A frequently used invocation is with $x_1 < x_2$, i.e. thinking of them as a range $[x_1, x_2]$

- ChainableArray.lincurve uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(0, 1, 100)) # turn data into ChainableArray
xs.plot("k-", label="input data") # plot input data
xs.lincurve(0, 1, 0, 0.4, 5).plot("r-", label="lincurve")
xs.lincurve(0.2, 0.5, 1, 0, 2.5, "minmax").plot("g-", label="lincurve with clip")
plt.legend(); plt.grid()

the following plot shows how the curve parameter influences the mapping function

In [ ]:
xs = chain(np.linspace(0, 100, 100))
for i, curve in enumerate(range(-9, 10, 3)):
    xs.lincurve(0, 100, -10, 10, curve).plot('-', label=f"curve={curve}")
plt.legend(fontsize=6); plt.grid();

### `curvelin`

- curvelin is implemented in analogy to the SC3 function
- `curvelin(v, x1, x2, y1, y2, curve=-2)` maps v (scalar or arraylike) from an assumed curve-exponential input range [x1, x2] to a linear output range [y1, y2] using the following function (with c=curve): 
$$ f(x) = y_1 + \frac{y_2 - y_1}{c}\log\left(\frac{a + x_1 - x}{a}\right)  ~~~\text{with}~~~ a = \frac{x_2 - x_1}{1 - e^c}$$

- This is the opposite transformation to `lincurve`.
- note that this `curvelin` function extrapolates by default.
- clipping can be controlled via the clip argument (values None (default), `"min"`, `"max"`, or anything else for minmax clipping.
- A frequently used invocation is with $x_1 < x_2$, i.e. thinking of them as a range $[x_1, x_2]$

- ChainableArray.curvelin uses self as input.
- Here are some mapping examples and plots
- The first shows how curvelin unmaps or reverts a lincurve warped interval when using the same curve argument.

In [ ]:
xs = chain(np.linspace(0, 1, 100)) # turn data into ChainableArray
xs.plot("k-", label="input data for lincurve") # plot input data
curve = 10
ys = xs.lincurve(0, 1, 0, 0.6, curve).plot("r-", label="lincurve output")
xs = ys.curvelin(0, 0.6, 0, 1, curve).plot("b-.", lw=3, alpha=0.5, label="curvelin to undo lincurve")
plt.legend(); plt.grid()

the following plot shows how the curve parameter influences the mapping function

In [ ]:
xs = chain(np.linspace(0, 100, 500))
for i, curve in enumerate(range(-9, 10, 3)):
    xs.curvelin(0, 100, -10, 10, curve).plot('-', label=f"curve={curve}")
plt.legend(fontsize=6); plt.grid();

### `bilin`

- bilin is implemented similar to the SC3 function, yet with an API change:
- `bilin(v, xcenter, xmin, xmax, ycenter, ymin, ymax)` maps v (scalar or arraylike) 
  according to two linear segments:
  - [xmin, xcenter] to [ymin, ycenter] with default extrapolation beyond xmin
  - [xcenter, xmax] to [ycenter, ymax] with default extrapolation beyond xmax
- this mapping is achieved using `pyamapping.interp_spline()`.
- kwargs are passed on to `interp_spline()` if needed.
- in case, no extrapolation is wanted, pyampapping.interp() offers an alternative
  with a different interface.

- ChainableArray.bilin uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.arange(0, 100)) # turn data into ChainableArray
xs.bilin(60, 20, 80, 0, -20, 60).plot("b.", ms=1, label='bilin');
for (x, y, t) in [[20, -20, "(xmin, ymin)"], [80, 60, "(xmax, ymax)"], [60, 0, "(xcenter, ycenter)"]]:
    plt.plot([x], [y], "ro")
    plt.text(x+1, y-5, t, fontsize=6)
plt.legend(); plt.grid()

### `clip`

- clip is implemented in analogy to the SC3 clip
- `clip(value, minimum, maximum)` clips value (scalar or arraylike) to a certain range [minimum, maximum]
- default values for minium and maximum are so that no clipping occurs, i.e. specifying only minimum or maximum allows one-sided clipping.

- ChainableArray.clip uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(0, 1, 100)) # turn data into ChainableArray
xs.plot("k:", label="input data") # plot input data
xs.clip(0.2, 0.7).plot("r-", label="linlin with clip")
plt.legend(); plt.grid()

In [ ]:
# plot mapping function (output vs input)
ys = xs.clip(0.25, 0.75)
plt.figure(figsize=(3, 1.5)); plt.plot(xs, ys, label=''); plt.grid(); 

### `midi_to_cps`

- midi_to_cps is implemented in analogy to the SC3 midicps function
- the shorter (less pythonic name) midicps can be used as well
- `midi_to_cps(midi_note)` converts MIDI note midi_note (value or arraylike) to cycles per second (aka Hz).
- The mapping function is
    $$ f(x) = 440\cdot 2^{\frac{x-69}{12}} $$
  which obviously maps MIDI 69 to 440 Hz, the reference for definition. 

In [ ]:
pam.midi_to_cps(69+12) # should be 880, i.e. one octave above MIDI 69

In [ ]:
chain(np.arange(60, 74, 2)).midicps().round(1) # rounded frequencies of full tone scale ('c-d-e-f#-g#-a#-c')

- ChainableArray.midi_to_cps uses self as input.
- Here a mapping examples and plots

In [ ]:
xs = chain(np.arange(21, 109)) # the standard MIDI range
xs.plot("k-", label="input data (MIDI notes)") # plot input data
xs.midi_to_cps().plot("r-", label="midi_to_cps result [in Hz]")
plt.legend(); plt.grid();

## Tour of Mapping Functions: additional/novel mapping functions

- The following function are new additions (i.e. there is no counterpart in Supercollider3 resp. librosa).
- For the following demonstrations, we use `xs` to refer to the input array and `ys` for the outputs.

### `linpoly`

- linpoly has no corresponding SC3 function, it provides a polynomial mapping
- `linpoly(v, xmax, y1, y2, curve=2, clip)` maps v (scalar or arraylike) from an assumed linear input range [-xmax, xmax] to an output range [y1, y2] using the following polynomial mapping function, using a polynomial order m

$$ m = \left\{\begin{align*} 1 + \text{curve}         & ~~~\text{if} & \text{curve} \ge 0\\
                     \frac{1}{1-\text{curve}} & ~~~\text{else} & 
        \end{align*} \right.
$$
using the mapping function
$$ f(x) = y_1 + \frac{y_2 - y_1}{2} \cdot \left(1 + \text{sign}(x) \left|\frac{x}{x_{\max}}\right|^m\right)$$

- note that np.sign is used
- note that this `linpoly` function extrapolates by default.
- clipping can be controlled via the clip argument (values: None as default, `"min"`, `"max"`, or anything else for minmax clipping.)
- It can be used to provide a sensitivity magnification (or reduction) around 0, the center of the input interval.

- ChainableArray.linpoly uses self as input.
- Here are some mapping examples and plots

In [ ]:
xs = chain(np.linspace(-3, 3, 200)) # turn data into ChainableArray
xs.plot("k-", label="input data for linpoly") # plot input data
xs.linpoly(3, 0, 20, curve=1).plot("r-", label="linpoly output")
xs.linpoly(3, 0, 20, curve=-1).plot("b-", label="linpoly output")
plt.legend(); plt.grid()

the following plot shows how the curve parameter influences the mapping function

In [ ]:
xs = chain(np.linspace(-1, 1, 200))
for i, curve in enumerate(range(-3, 3, 1)):
    xs.linpoly(1, -10, 10, curve).plot('-', label=f"curve={curve}")
plt.legend(fontsize=6); plt.grid();

### `interp_spline`

- interp_spline provides an interface to  `scipy.interpolate.make_interp_spline` for line segment interpolation.
- `interp_spline(v, xc, yc, k)` maps v (scalar or arraylike) along the spline 
  - defined by the input coordinates in array xc 
  - and corresponding output coordinates in yc
  - using interpolation order k (default 1=linear)
- note that interp_spline extrapolation beyond segments.
- interp_spline is called from `bilin()`.

In [ ]:
xs = chain(np.linspace(0, 8.2, 1000)) # turn data into ChainableArray
xc = [1, 2, 5, 7, 8]
yc = [3, 1, 2, 9, 1]
plt.plot(xc, yc, "ro", label="given points")
for k in [0, 1, 2]:
    ys = xs.interp_spline(xc, yc, k=k)
    plt.plot(xs,ys, label=f"spline with k={k}")
plt.legend(fontsize=7); plt.grid()

### `interp`

- interp provides an interface to  `np.interp` for line segment interpolation.
- `interp(v, xc, yc)` maps v (scalar or arraylike) along the sample points 
  - defined by the input coordinates in array xc (monotonically increasing)
  - and corresponding output coordinates in yc
- note that interp clips beyond xc limits.
- interp can be used as alternative to bilin in case clipped output is needed.
  - some may prefer the more tidy API with x and y values in their arrays.

In [ ]:
xs = chain(np.linspace(0, 8.2, 1000)) # turn data into ChainableArray
xc = [1, 2, 5, 7, 8]
yc = [7, 1, 2, 9, 1]
plt.plot(xc, yc, "ro", ms=3, label="given points")
ys = xs.interp(xc, yc)
plt.plot(xs, ys, "b,", label=f"interp")
plt.legend(fontsize=7); plt.grid()

In [ ]:
xs = chain(np.arange(-100, 150)) # turn data into ChainableArray
# use interp if clipping is wanted (faster for small datasets)
xs.interp([-50, 50, 100],[0, 0.4, 1]).plot("r-.", label="interp with clipping"); 
xs.bilin(50, -50, 100, 0.4, 0, 1).plot("b:", label='bilin with extrapolation');
plt.legend();plt.grid();